### Importing libraries and Function definitions

In [23]:
from selenium import webdriver      # !pip install selenium
from bs4 import BeautifulSoup       # !pip install BeautifulSoup
import requests                     # !pip install requests
from pyuca import Collator          # !pip install pyuca
import time
import pandas as pd

In [24]:
def reject_cookies():

    # Find the 'Reject All' button and press it
    button = driver.find_element("xpath", "//button[contains(text(), 'Reject All')]") 
    button.click()

In [25]:
def scrape_teacher_links(): 

    # List to store the extracted links
    link_list = []

    # Get the page source 
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')

    # Find all the teacher cards
    links = soup.find_all('a', class_='card-avatar__link arrow-animate')

    # Loop through each <a> tag found and extract the href attribute 
    for link in links:
        # Get the href attribute
        profile_url = link.get('href')
        
        # Append the URL to the list
        link_list.append(profile_url)
        
    return link_list

In [26]:
def scrape_teacher_info_selenium():                  

    # Get the page source 
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')

    # Find and extract all the needed data
    name = soup.find('h1', class_='card-avatar__heading gamma').text                                                    # Get the professor name
    title = str(soup.find('h2', class_='card-avatar__posheading').text).replace('\n', '')                               # Get the title + remove \n 
    courses = ', '.join([el.text for el in soup.find_all('li', class_='card-avatar__list-item')]).replace('\n', '')     # Get all the course names + remove \n
    biography = str(soup.find('div', class_='content__copy').get_text())                                                # Get the biography

    # Get the number of publications
    headings = soup.find_all('h2', class_='search-results__heading gamma')
    n_publications = sum(1 for heading in headings if heading.get_text(strip=True))


    return [name, title, courses, n_publications, biography]

In [27]:
def scrape_teacher_info_requests(teacher_link):

    # Get the page source 
    r = requests.get(teacher_link)
    soup = BeautifulSoup(r.text, 'html.parser')

    # Find and extract all the needed data
    name = soup.find('h1', class_='card-avatar__heading gamma').text                                                    # Get the professor name
    title = str(soup.find('h2', class_='card-avatar__posheading').text).replace('\n', '').replace('\r', '')             # Get the title + remove \n and \r 
    courses = ', '.join([el.text for el in soup.find_all('li', class_='card-avatar__list-item')]).replace('\n', '')     # Get all the course names + remove \n
    biography = str(soup.find('div', class_='content__copy').text)                                                      # Get the biography

    # Get the number of publications
    headings = soup.find_all('h2', class_='search-results__heading gamma')
    n_publications = sum(1 for heading in headings if heading.get_text(strip=True))
    

    return [name, title, courses, n_publications, biography]

### Webscraping and Data exporting

In [28]:
# Open the driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# Navigate to URL
url = "https://www.novaims.unl.pt/en/nova-ims/teaching-staff/"
driver.get(url)
print('Connection with the web interface established.')

# Wait for the page to load completely, MAY NEED ADJUSTING ! ! !
time.sleep(3)

# Reject cookies banner
reject_cookies()
print('External Cookies rejected.')

# List to store all teacher links across pages
all_teacher_links = []

# Navigate through all the pages, extracting the links
print("Initiating teachers' resource links extraction ...")
while True:

    # Scrape all the links on the page
    all_teacher_links.extend(scrape_teacher_links())

    # Find the 'Next page' button
    next_button = driver.find_element("xpath", "//button[@aria-label='Página Seguinte']")
    
    if next_button.get_attribute("disabled") == "true":
        print(f"Resource collection completed. {len(all_teacher_links)} links found.")
        break                   # If button is disabled -> we reached the last page -> break the loop
    
    # If button is active, click it to navigate to the next page
    next_button.click() 
    
    # Wait for the page to load completely, MAY NEED ADJUSTING ! ! !
    time.sleep(2)

# Close the browser, because request does not need it
driver.quit()
print('Browser interface closed.')


# Prepare the Datebase layout
teachers_database = pd.DataFrame(columns=['Name', 'Title', 'Courses', 'Publications', 'Biography'])

print("Retrieving professors' professional data ...")

# Navigate through all the collected links
for link in set(all_teacher_links):                 # Remove any duplicate links, if such exist
    
    # driver.get(link)                                                                          #   ╮             
    #                                                                                           #   |
    # Wait for the page to load completely, MAY NEED ADJUSTING ! ! !                            #   |
    # time.sleep(0.5)                                                                           #   |>  Scraping using selenium
    #                                                                                           #   |
    # Extract all the needed data                                                               #   |
    # teachers_database.loc[len(teachers_database)] = scrape_teacher_info_selenium()            #   ╯


    # Extract all the needed data
    teachers_database.loc[len(teachers_database)] = scrape_teacher_info_requests(link)

print("All relative data packets secured.")


# Prints are there just because it looks cool

Connection with the web interface established.
External Cookies rejected.
Initiating teachers' resource links extraction ...
Resource collection completed. 231 links found.
Browser interface closed.
Retrieving professors' professional data ...
All relative data packets secured.


In [29]:
# Special sort for names
# Because default sort does not work with portuguese letters

collator = Collator()

teachers_database['sort_key'] = teachers_database['Name'].apply(lambda x: collator.sort_key(x))     # Generate sort keys
teachers_database = teachers_database.sort_values('sort_key')                                       # Sort the DataFrame by the sort keys
teachers_database = teachers_database.drop(columns=['sort_key']).reset_index(drop=True)             # Drop the 'sort_key' column

teachers_database.to_csv('teachers_database.csv', index=False)                                      # Export the Database

In [30]:
teachers_database.head()

,Name,Title,Courses,Publications,Biography
0,Afonso Malheiro,Adjunct Lecturer,Web Analytics,0,Bachelor's degree in Management (Catholic Univ...
1,Afshin Ashofteh,Invited Assistant Professor,"Banking and Insurance Economics, Credit Risk S...",21,Afshin Ashofteh is a full-time University Prof...
2,Alexandra Variz,Adjunct Lecturer,Digital Marketing & E-Commerce,0,"Currently, I am working as a Digital Marketing..."
3,Alexandre Guilherme Marques,Adjunct Lecturer,Deep Learning,0,Alexandre Marques is an assistant professor at...
4,Alexandre Neto,Adjunct Lecturer,Group Project Seminar on Programming and Analysis,0,Bachelor's degree in Geographical Engineering ...


In [11]:
teachers_database.shape

(231, 5)

### Database comparison 

In [21]:
def compare_dataframes(df1, df2):
    similarity = {}
    
    # Check if shapes are the same
    if df1.shape != df2.shape:
        similarity['shape'] = False
        return "DataFrames have different shapes. Comparison not possible."
    else:
        similarity['shape'] = True

    # Check if columns are the same
    similarity['columns'] = df1.columns.equals(df2.columns)

    # Compare cell-by-cell and calculate similarity
    comparison = df1 == df2
    identical_values = comparison.values.sum()
    total_values = comparison.size
    similarity['similarity_ratio'] = identical_values / total_values

    # Provide summary
    return {
        "Shape identical": similarity['shape'],
        "Columns identical": similarity['columns'],
        "Cell similarity ratio": similarity['similarity_ratio']
    }

In [ ]:
# example_df = pd.read_csv('nova_ims_teaching_staff_2024-11-21.csv').fillna('')

# compare_dataframes(teachers_database, example_df)

{'Shape identical': True,
 'Columns identical': True,
 'Cell similarity ratio': 1.0}